# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', '')}: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All elements are referenced by their `@id`.

In [ ]:
from pprint import pprint

# Print all record sets (@id, name, description)
record_sets = list(dataset.metadata.recordSets)
print(f"Found {len(record_sets)} record sets in the dataset.\n")
record_set_ids = [r['@id'] for r in record_sets]
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(none)')}")
    print(f"  Description: {rs.get('description', '(none)')}")
    # List columns (fields) for this record set
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print(f"  Fields in this record set:")
        for field in fields:
            print(f"    Field @id: {field['@id']}, name: {field.get('name', '(none)')}, dataType: {field.get('dataType', '(none)')}")
    print("")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id
dataframes = dict()
for rs in record_sets:
    record_set_id = rs['@id']
    print(f"Loading data for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"- Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# Use the first record set for demonstration below
main_record_set_id = record_sets[0]['@id'] if record_sets else None
if main_record_set_id:
    print(f'Columns in {main_record_set_id}:')
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Choose a numeric field for analysis
import numpy as np
rs = dataset.metadata.recordSets[0] if dataset.metadata.recordSets else None
if rs is not None:
    fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
    # Find a field with dataType Float or Integer
    numeric_fields = [f for f in fields if f.get('dataType','').lower() in ('float', 'integer', 'number')]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]['@id']
        numeric_field_name = numeric_fields[0].get('name', numeric_field_id)
        print(f"Using numeric field: {numeric_field_id} ({numeric_field_name})\n")
        df = dataframes[rs['@id']]
        # Ensure the field is numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in {rs['@id']} where {numeric_field_id} > mean ({threshold:.3f}):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field (choose first non-numeric field)
        cat_fields = [f for f in fields if f.get('dataType','').lower() not in ('float', 'integer', 'number')]
        if cat_fields:
            group_field_id = cat_fields[0]['@id']
            group_field_name = cat_fields[0].get('name', group_field_id)
            if group_field_id in df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index(name=f"mean_{numeric_field_id}")
                print(f"\nGrouped data by {group_field_id} ({group_field_name}):")
                print(grouped_df.head())
    else:
        print('No numeric fields found in the first record set.')
else:
    print('No record sets found in the dataset.')


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs is not None and numeric_fields:
    num_field = numeric_field_id
    df = dataframes[rs['@id']]
    plt.figure(figsize=(8,4))
    sns.histplot(df[num_field].dropna(), kde=True)
    plt.title(f'Distribution of {num_field}')
    plt.xlabel(num_field)
    plt.tight_layout()
    plt.show()
    # If has a categorical group field, draw boxplot
    if cat_fields and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[num_field])
        plt.title(f'{num_field} by {group_field_id}')
        plt.ylabel(num_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook provided an outline for using the `mlcroissant` library to programmatically explore, extract, and analyze structured, FAIR-compliant datasets.
- Key entities such as record sets, fields (columns), and data types were referenced using their unique `@id` in accordance with the Croissant schema best practices.
- Using simple EDA and visualization, one can begin to reveal aggregate properties and distributions of the data for scientific or ML use cases.
- For more detailed analysis, refer to the dataset's full documentation at the provided Croissant schema URL.